# Model Throughput

At inference time, count the number of floating point operations (FLOPs) required to process a single input sample.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from utils.const import SEED
from data_modules.acdc import ACDC3DDataModule, ACDC3DWithPredictedMaskDataModule, ACDCMaskDataModule
from utils.utils import get_data, setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
# data_module = ACDC3DWithPredictedMaskDataModule()
data_module = ACDC3DDataModule()

# Reseed after preprocessing data
L.seed_everything(SEED)

loader_test = data_module.test_dataloader()


In [ ]:
from arch.nvae_corrector.nvae_corrector import NVAECorrector
from arch.vae_corrector.vae_corrector import VAECorrector

nvae_m2m_extended = "logs/nvae_corrector_acdc/default-extended/checkpoints/epoch=20-step=17976.ckpt"
vae_m2m = "logs/vae_corrector_acdc/best/checkpoints/epoch=40-step=4387.ckpt"

# To use ACUNet, change in ACUNet class to:
# self.nvae = NVAE.load_from_checkpoint(nvae_path, map_location="meta")
# # self.nvae.freeze() <- comment this line
nvae_acunet = "logs/unet_acdc/default-shape-prior/checkpoints/epoch=47-step=2592.ckpt"

cnvae_i2m = "logs/cnvae_acdc/best/checkpoints/epoch=83-step=17976.ckpt"

unet = "logs/unet_acdc/segmentation-models/unet/checkpoints/epoch=33-step=1836.ckpt"
attention_unet = "logs/unet_acdc/segmentation-models/attentionunet/checkpoints/epoch=30-step=1674.ckpt"
resunet = "/Users/freddy/Unshared/nvae-shape-encoding/logs/unet_acdc/segmentation-models/resunet/checkpoints/epoch=24-step=1350.ckpt"
swinunet = "/Users/freddy/Unshared/nvae-shape-encoding/logs/unet_acdc/segmentation-models/swinunet/checkpoints/epoch=40-step=2214.ckpt"
high_resnet = "/Users/freddy/Unshared/nvae-shape-encoding/logs/unet_acdc/high-resnet/checkpoints/epoch=97-step=5292.ckpt"


In [ ]:
from lightning.fabric.utilities.throughput import measure_flops

from arch.cnvae.cnvae import CNVAE
from arch.unet.acunet import ACUNet
from arch.unet.attentionunet import AttentionUNet
from arch.unet.resunet import ResUNet
from arch.unet.swinunet import SwinUNetR
from arch.unet.unet import UNet
from arch.vae_corrector.vae_corrector import VAECorrector
from arch.unet.high_resnet import HighResnet

# Uncomment this for SwinUNetR

# model = SwinUNetR.load_from_checkpoint(swinunet)
# model.to("meta")

with torch.device("meta"):
    # model = NVAECorrector.load_from_checkpoint(nvae_m2m_extended, map_location="meta")
    # model = VAECorrector.load_from_checkpoint(vae_m2m, map_location="meta")
    # model = ACUNet.load_from_checkpoint(nvae_acunet, map_location="meta")
    # model = CNVAE.load_from_checkpoint(cnvae_i2m, map_location="meta")
    # model = UNet.load_from_checkpoint(unet, map_location="meta")
    # model = AttentionUNet.load_from_checkpoint(attention_unet, map_location="meta")
    # model = ResUNet.load_from_checkpoint(resunet, map_location="meta")
    model = HighResnet.load_from_checkpoint(high_resnet, map_location="meta")

flops_acc = 0

for batch in loader_test:
    x, y, _, _ = batch
    x = x.squeeze(0)
    y = y.squeeze(0)
    
    x = x.to("meta")
    y = y.to("meta")
    
    model_inference = lambda: model(x)
    flops = measure_flops(model, model_inference)
    print(f"FLOPS: {flops}")
    
    flops_acc += flops

print(f"Total FLOPS: {flops_acc}")
print(f"Average FLOPS per slice: {flops_acc / 1076}")
